# Data Cleaning Perfect - Commercial Vente Marrakech
This notebook implements the 'Perfect Cleaning' pipeline for Machine Learning preparation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# File path
file_path = '../../data/marrakech_immo_vente/commercial_vente.csv'

# Load data
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}")
    print(f"Initial shape: {df.shape}")
    display(df.head())
else:
    print(f"ERROR: File not found at {file_path}")

Successfully loaded ../../data/marrakech_immo_vente/commercial_vente.csv
Initial shape: (1, 34)


,id,titre,prix,localisation,type_bien,surface,chambres,salles_bain,description,agence,...,etage,surface_terrain,prix_num,surface_num,chambres_num,salles_bain_num,nb_pieces,quartier,prix_m2,prix_m2_median_quartier
0,VE26019,Local commercial Idéal pour café-restaurant au...,8 500 000 Dhs,Marrakech,Commercial,m²,NaN,NaN,Un spacieux local commercial de 283 m² + 263 m...,Marrakech Realty,...,-1,NaN,8500000.0,NaN,0.0,0.0,1.0,Autre,NaN,NaN


## 1. Target Protection & Deduplication
We must drop rows without a price (target) and remove duplicates.

In [2]:
if 'df' in locals():
    # 1. Drop missing target
    if 'prix_num' in df.columns:
        initial_len = len(df)
        df = df.dropna(subset=['prix_num'])
        print(f"Dropped {initial_len - len(df)} rows with missing prix_num")
    
    # 2. Drop duplicates
    initial_len = len(df)
    df = df.drop_duplicates()
    print(f"Dropped {initial_len - len(df)} duplicate rows")

Dropped 0 rows with missing prix_num
Dropped 0 duplicate rows


In [ ]:
# Conversion des prix en DH si nécessaire
if 'df' in locals():
    if 'prix' in df.columns and 'prix_num' in df.columns:
        # Taux de conversion approximatif (1 EUR = 10.8 MAD)
        EUR_TO_MAD = 10.8
        
        def convert_to_dh(row):
            price_str = str(row['prix']).lower() if pd.notna(row['prix']) else ''
            price_num = row['prix_num']
            
            # Si le prix contient l'euro et qu'on n'a pas encore une valeur en DH cohérente
            if '€' in price_str or 'eur' in price_str:
                import re
                # Extraire le montant en euro (souvent le premier nombre)
                nums = re.findall(r'\d+', price_str.replace(' ', '').replace(',', ''))
                if nums:
                    # Si c'est en euros, on multiplie par notre taux
                    euro_val = float(nums[0])
                    # On vérifie si price_num est déjà une conversion correcte (proche de euro_val * 10)
                    # Sinon on force la conversion
                    if pd.isna(price_num) or price_num < euro_val * 5:
                        return round(euro_val * EUR_TO_MAD)
            
            return price_num
            
        df['prix_num'] = df.apply(convert_to_dh, axis=1)
        print("Prix uniformisés en DH.")

## 2. Feature Selection & Data Leakage Prevention
Drop non-predictive columns and columns that cause data leakage (like prix_m2).

In [3]:
if 'df' in locals():
    cols_to_drop = ['id', 'url', 'source', 'titre', 'description', 'prix', 'surface', 'localisation', 'chambres', 'salles_bain', 'prix_m2']
    existing_drops = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=existing_drops)
    print(f"Dropped columns: {existing_drops}")
    print(f"Remaining columns: {df.columns.tolist()}")

Dropped columns: ['id', 'url', 'source', 'titre', 'description', 'prix', 'surface', 'localisation', 'chambres', 'salles_bain', 'prix_m2']
Remaining columns: ['type_bien', 'agence', 'piscine', 'parking', 'ascenseur', 'terrasse', 'jardin', 'climatisation', 'securite', 'vue', 'meuble', 'neuf', 'cave', 'hammam', 'etage', 'surface_terrain', 'prix_num', 'surface_num', 'chambres_num', 'salles_bain_num', 'nb_pieces', 'quartier', 'prix_m2_median_quartier']


## 3. Intelligent Imputation
Handling missing values for features.

In [4]:
if 'df' in locals():
    # Missing values before
    print("Missing values before imputation:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    
    # 1. Numeric: Median
    if 'surface_num' in df.columns:
        df['surface_num'] = df['surface_num'].fillna(df['surface_num'].median())
    
    # 2. Rooms/Baths logic
    non_residential = ['terrain', 'locaux', 'commercial', 'bureaux']
    if 'commercial' in non_residential:
        if 'chambres_num' in df.columns: df['chambres_num'] = df['chambres_num'].fillna(0.0)
        if 'salles_bain_num' in df.columns: df['salles_bain_num'] = df['salles_bain_num'].fillna(0.0)
    else:
        if 'chambres_num' in df.columns: df['chambres_num'] = df['chambres_num'].fillna(round(df['chambres_num'].mean() if not df['chambres_num'].isnull().all() else 0))
        if 'salles_bain_num' in df.columns: df['salles_bain_num'] = df['salles_bain_num'].fillna(round(df['salles_bain_num'].mean() if not df['salles_bain_num'].isnull().all() else 0))
            
    # 3. Categorical: Mode
    cat_cols = ['agence', 'type_bien', 'quartier']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Inconnu')

    print("\nMissing values after imputation:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before imputation:
surface_terrain            1
surface_num                1
prix_m2_median_quartier    1
dtype: int64

Missing values after imputation:
surface_terrain            1
surface_num                1
prix_m2_median_quartier    1
dtype: int64


## 4. Feature Engineering
Applying log transformations to stabilize variance.

In [5]:
if 'df' in locals():
    if 'prix_num' in df.columns:
        df['log_prix_num'] = np.log1p(df['prix_num'])
        print("Created log_prix_num")
    
    if 'surface_num' in df.columns:
        df['log_surface_num'] = np.log1p(df['surface_num'])
        print("Created log_surface_num")

Created log_prix_num
Created log_surface_num


## 5. Outlier Detection & Removal
Using 1st and 99th percentiles.

In [6]:
if 'df' in locals():
    for col in ['prix_num', 'surface_num']:
        if col in df.columns:
            q_low = df[col].quantile(0.01)
            q_hi  = df[col].quantile(0.99)
            df = df[(df[col] <= q_hi) & (df[col] >= q_low)]
            print(f"Filtered {col} between {q_low:.2f} and {q_hi:.2f}")
    
    print(f"Final shape: {df.shape}")

Filtered prix_num between 8500000.00 and 8500000.00
Filtered surface_num between nan and nan
Final shape: (0, 25)


## 6. Exporting ML-Ready Data

In [7]:
if 'df' in locals():
    output_dir = '../../data/cleaned_data/vente'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    output_path = os.path.join(output_dir, 'commercial_cleaned.csv')
    df.to_csv(output_path, index=False)
    print(f"Success! ML-ready data saved to {output_path}")

Success! ML-ready data saved to ../../data/cleaned_data/vente/commercial_cleaned.csv
